# Notebook 02: Oracle Reasoning Trace Distillation
**Teacher Model:** `deepseek-flash` via official API (`https://api.deepseek.com`)  
**Features:** Multithreaded extraction, native `reasoning_content` retrieval, token length budgeting, numeric validation, and target dataset formatting.


In [2]:
import os
import json
import re
import argparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv("../.env")

RAW_DATA_PATH = "../data/raw/gsm8k_train_5k.json"
PROCESSED_DIR = "../data/processed"
MAX_TRACE_TOKENS = 1500  # Thesis length constraint for 12GB VRAM
os.makedirs(PROCESSED_DIR, exist_ok=True)


### 1. Regex Extraction & Token Estimator


In [3]:
def extract_numeric_value(text):
    """Extract numeric value from answer text (handles \boxed{}, decimals, and plain numbers)."""
    if not text:
        return None
    boxed_match = re.search(r"\\boxed\{([0-9\.,]+)\}", text)
    if boxed_match:
        return boxed_match.group(1).replace(",", "").strip()
    
    numbers = re.findall(r"[-+]?\d*\.?\d+", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return None

def estimate_tokens(text):
    if not text:
        return 0
    return int(len(text.split()) * 1.3)


### 2. DeepSeek Flash Request Handler


In [4]:
def process_single_problem(client, sample):
    """Call DeepSeek API (V4 Flash) for a single GSM8K problem."""
    question = sample["question"]
    target_numeric = sample["target_answer"]

    system_prompt = (
        "You are an expert mathematical reasoner. Solve the problem by showing your step-by-step derivations.\n"
        "Conclude your final answer clearly at the end with: 'The final answer is \\boxed{number}'."
    )

    try:
        response = client.chat.completions.create(
            model="deepseek-flash",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question}
            ],
            stream=False,
            reasoning_effort="high",
            extra_body={"thinking": {"type": "enabled"}}
        )

        message = response.choices[0].message
        reasoning_trace = getattr(message, "reasoning_content", "") or ""
        final_answer_text = message.content or ""

        # 1. Length Filter
        trace_token_len = estimate_tokens(reasoning_trace)
        if trace_token_len > MAX_TRACE_TOKENS or trace_token_len == 0:
            return None, "Length_Filtered"

        # 2. Validation Filter
        extracted_numeric = extract_numeric_value(final_answer_text)
        if not extracted_numeric or str(extracted_numeric) != str(target_numeric):
            return None, "Validation_Filtered"

        return {
            "id": sample["id"],
            "question": question,
            "reasoning_trace": reasoning_trace,
            "final_answer": final_answer_text,
            "target_answer": target_numeric
        }, "Success"

    except Exception as e:
        return None, f"API_Error: {e}"


### 3. Run Concurrent Distillation Pipeline


In [5]:
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("DEEPSEEK_API_KEY not found in environment or ../.env!")

client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

with open(RAW_DATA_PATH, "r", encoding="utf-8") as f:
    gsm8k_data = json.load(f)

# Set limit=None for full run, or an integer for pilot testing
TEST_LIMIT = 20
if TEST_LIMIT:
    print(f"Running on first {TEST_LIMIT} samples for testing...")
    gsm8k_data = gsm8k_data[:TEST_LIMIT]

validated_results = []
stats = {"Success": 0, "Length_Filtered": 0, "Validation_Filtered": 0, "API_Error": 0}

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(process_single_problem, client, item): item for item in gsm8k_data}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Distilling Reasoning"):
        result, status = future.result()
        if status.startswith("API_Error"):
            stats["API_Error"] += 1
        else:
            stats[status] += 1
        
        if result:
            validated_results.append(result)

print("\n" + "=" * 60)
print(f"Total Processed:       {len(gsm8k_data)}")
print(f"✓ Validated Samples:   {stats['Success']}")
print(f"✗ Failed Validation:   {stats['Validation_Filtered']}")
print(f"✗ Exceeded Token Limit:{stats['Length_Filtered']}")
print(f"✗ API Errors:          {stats['API_Error']}")


Running on first 20 samples for testing...


Distilling Reasoning: 100%|██████████| 20/20 [00:08<00:00,  2.39it/s]


Total Processed:       20
✓ Validated Samples:   20
✗ Failed Validation:   0
✗ Exceeded Token Limit:0
✗ API Errors:          0


### 4. Format and Save Fine-Tuning Datasets


In [6]:
monolithic_set = []
thinker_set = []
solver_set = []

for item in validated_results:
    q = item["question"]
    cot = item["reasoning_trace"]
    ans = item["final_answer"]

    # 1. Monolithic Control
    monolithic_set.append({
        "instruction": "Solve the following math problem by thinking step-by-step inside <think>...</think> tags, then provide the concise final answer.",
        "input": q,
        "output": f"<think>\n{cot}\n</think>\n{ans}"
    })

    # 2. Thinker Adapter
    thinker_set.append({
        "instruction": "Generate a step-by-step reasoning trace to solve the following math problem.",
        "input": q,
        "output": cot
    })

    # 3. Solver Adapter (Hard Bottleneck: CoT -> Answer)
    solver_set.append({
        "instruction": "Based on the provided reasoning trace, output the concise final answer.",
        "input": f"{cot}",
        "output": ans
    })

paths = {
    "baseline_monolithic.json": monolithic_set,
    "thinker_only.json": thinker_set,
    "solver_only.json": solver_set
}

for filename, data in paths.items():
    out_path = os.path.join(PROCESSED_DIR, filename)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"✓ Saved {len(data)} items to {out_path}")


✓ Saved 20 items to ../data/processed/baseline_monolithic.json
✓ Saved 20 items to ../data/processed/thinker_only.json
✓ Saved 20 items to ../data/processed/solver_only.json
